# Install libraries

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import io
import statsmodels.api as sm
from scipy.optimize import curve_fit
from scipy.stats import ttest_ind, chi2_contingency

from pathlib import Path

# Opening cleaned PAROS dataset

In [12]:
CURRENT_DIRECTORY = Path.cwd().resolve()

# Find project root that contains datasets
PROJECT_ROOT = next(
    p for p in [CURRENT_DIRECTORY, *CURRENT_DIRECTORY.parents]
    if (p / "datasets").exists()
)

CLEANED_DATASET_PATH = PROJECT_ROOT / "datasets" / "PAROS_Dataset_Cleaned.csv"

if not CLEANED_DATASET_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEANED_DATASET_PATH}")

df = pd.read_csv(CLEANED_DATASET_PATH)
print(f"Loaded cleaned PAROS dataset: {df.shape}")
display(df.head(3))

Loaded cleaned PAROS dataset: (2076, 71)


,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Age,Age Modifier,Gender,Race,...,Outcome of patient,Patient status,Date of Discharge or Death,Patient neurological status - Cerebral,Patient neurological status - Overall,Patient neurological status - Unknown,Year,Call_Time,Shock_Time,Time_to_Defib
0,Ems,2014-01-01,238889.0,NaN,Transport Center,Dhoby Ghaut Mrt Level B1,59,Years,Male,Chinese,...,Died In Ed,NaN,NaN,5.0,NaN,NaN,2014,2026-04-06 22:28:12,2026-04-06 22:39:17,11.083333
1,Ems,2014-01-05,272018.0,NaN,Public/Commercial Building,Level 2,66,Years,Male,Chinese,...,Died In Ed,NaN,NaN,5.0,NaN,NaN,2014,2026-04-06 15:00:42,2026-04-06 15:16:49,16.116667
2,Ems,2014-01-07,760105.0,NaN,Street/Highway,Level 1,80,Years,Male,Indian,...,Admitted,Remains In Hospital At 30Th Day Post Arrest,NaN,4.0,4.0,NaN,2014,2026-04-06 12:05:46,2026-04-06 12:14:08,8.366667


In [13]:
print(df.columns.tolist())


['Patient brought in by', 'Date of Incident', 'Location of incident', 'Location Unknown', 'Location Type', 'Location Type Other', 'Age', 'Age Modifier', 'Gender', 'Race', 'Time call received at dispatch center', 'No First Responder dispatched', 'Time First Responder dispatched', 'Time Ambulance dispatched', 'Time First Responder arrived at scene', 'Time Ambulance arrived at scene', 'Time EMS arrived at patient side', 'Time Ambulance arrived at ED', 'Arrest witnessed by', 'Bystander CPR', 'DA-CPR', 'First CPR initiated by', 'Bystander AED applied', 'Resuscitation attempted by EMS/Private ambulance', 'First arrest rhythm', 'Prehospital Defibrillation', 'Time of first shock given', 'Time of first shock Unknown', 'Defibrillation performed by - First Responder', 'Defibrillation performed by - Ambulance Crew', 'Defibrillation performed by - Bystander - Healthcare provider', 'Defibrillation performed by - Bystander - Lay Person', 'Defibrillation performed by - Bystander - Family', 'Other', 'R

# Pre-calcualte Operational Time Variables

In [14]:
# Convert timestamps to datetime objects to calculate time durations in minutes
call_time = pd.to_datetime(df['Time call received at dispatch center'], errors='coerce', format='mixed')
ems_time = pd.to_datetime(df['Time EMS arrived at patient side'], errors='coerce', format='mixed')
shock_time = pd.to_datetime(df['Time of first shock given'], errors='coerce', format='mixed')


df.loc[:, 'EMS_Response_Time'] = (ems_time - call_time).dt.total_seconds() / 60
df.loc[:, 'Time_to_First_Shock'] = (shock_time - call_time).dt.total_seconds() / 60

# Clean up negative times or absurd outliers
df.loc[df['EMS_Response_Time'] < 0, 'EMS_Response_Time'] = np.nan
df.loc[df['Time_to_First_Shock'] < 0, 'Time_to_First_Shock'] = np.nan

# Split the cohort betweeen with AED and without AED

In [15]:
df_aed = df[df['Bystander AED applied'].astype(str).str.contains('Yes', case=False, na=False)]
df_no_aed = df[~df['Bystander AED applied'].astype(str).str.contains('Yes', case=False, na=False)]

n_total = len(df)
n_aed = len(df_aed)
n_no_aed = len(df_no_aed)

# Helper functions for Calculating

In [16]:
def get_cont(col):
    """Calculates Mean ± SD and T-Test p-value for continuous variables."""
    def format_stat(data):
        s = pd.to_numeric(data[col], errors='coerce').dropna()
        if len(s) == 0: return "0.0 ± 0.0"
        return f"{s.mean():.1f} ± {s.std():.1f}"
    
    a = pd.to_numeric(df_aed[col], errors='coerce').dropna()
    b = pd.to_numeric(df_no_aed[col], errors='coerce').dropna()
    p_val = "-"
    if len(a) > 0 and len(b) > 0:
        _, p = ttest_ind(a, b, equal_var=False)
        p_val = f"{p:.3f}" if p >= 0.001 else "<0.001"
    return format_stat(df), format_stat(df_aed), format_stat(df_no_aed), p_val

def get_cat(col, regex, search_multiple_cols=False):
    """Calculates percentages for categorical matches."""
    def format_cat(data, n_group):
        if search_multiple_cols:
            s = data['Outcome of patient'].astype(str) + " " + \
                data['Patient status'].astype(str) + " " + \
                data['Final status at scene'].astype(str)
        else:
            s = data[col].astype(str)
        count = s.str.contains(regex, case=False, na=False, regex=True).sum()
        pct = (count / n_group * 100) if n_group > 0 else 0
        return f"{pct:.1f}%"
    return format_cat(df, n_total), format_cat(df_aed, n_aed), format_cat(df_no_aed, n_no_aed)

def get_missing(col):
    """Calculates missingness strictly avoiding double-counting 'nan' strings."""
    def format_missing(data, n_group):
        s = data[col]
        nan_count = s.isna().sum()
        text_count = s.dropna().astype(str).str.contains('^unknown$|^missing$', case=False, regex=True).sum()
        pct = ((nan_count + text_count) / n_group * 100) if n_group > 0 else 0
        return f"{pct:.1f}%"
    return format_missing(df, n_total), format_missing(df_aed, n_aed), format_missing(df_no_aed, n_no_aed)

def calc_chi2_pval(col, regexes):
    """Calculates Chi-Square p-values across groups."""
    counts_aed = [df_aed[col].astype(str).str.contains(r, case=False, na=False, regex=True).sum() for r in regexes]
    counts_no = [df_no_aed[col].astype(str).str.contains(r, case=False, na=False, regex=True).sum() for r in regexes]
    try:
        _, p, _, _ = chi2_contingency([counts_aed, counts_no])
        return f"{p:.3f}" if p >= 0.001 else "<0.001"
    except:
        return "-"

def get_multi_missing(cols):
    """Calculates missingness if ALL specified columns are NaN."""
    def format_missing(data, n_group):
        mask = data[cols].isna().all(axis=1)
        pct = (mask.sum() / n_group * 100) if n_group > 0 else 0
        return f"{pct:.1f}%"
    return format_missing(df, n_total), format_missing(df_aed, n_aed), format_missing(df_no_aed, n_no_aed)

def get_remainder_pct(main_pct_str):
    """Calculates 100% minus a percentage string."""
    pct = float(main_pct_str.strip('%'))
    return f"{100 - pct:.1f}%"

def get_remainder_of_two(p1_str, p2_str):
    """Calculates 100% minus two percentage strings."""
    p1 = float(p1_str.strip('%'))
    p2 = float(p2_str.strip('%'))
    return f"{max(0, 100 - p1 - p2):.1f}%"

# Calculate Metrics for Table 1 Rows

In [17]:
# --- 1. DEMOGRAPHICS ---
age_all, age_aed, age_no, age_p = get_cont('Age')
age_miss_all, age_miss_aed, age_miss_no = get_missing('Age')

gen_all, gen_aed, gen_no = get_cat('Gender', r'\bMale\b|^M$')
gen_miss_all, gen_miss_aed, gen_miss_no = get_missing('Gender')
gen_p = calc_chi2_pval('Gender', [r'\bMale\b|^M$', r'\bFemale\b|^F$'])

chi_all, chi_aed, chi_no = get_cat('Race', r'\bChinese\b')
mal_all, mal_aed, mal_no = get_cat('Race', r'\bMalay\b')
ind_all, ind_aed, ind_no = get_cat('Race', r'\bIndian\b')
eur_all, eur_aed, eur_no = get_cat('Race', r'\bEurasian\b')
oth_all, oth_aed, oth_no = get_cat('Race', r'\bOther\b|\bCaucasian\b|\bUnknown\b')
eth_p = calc_chi2_pval('Race', [r'\bChinese\b', r'\bMalay\b', r'\bIndian\b', r'\bEurasian\b', r'\bOther\b'])

# --- 2. ARREST CONTEXT ---
pub_all, pub_aed, pub_no = get_cat('Location Type', r'Public|Commercial|Work|Street|Recreation|Transport|Airport')
res_all, res_aed, res_no = get_cat('Location Type', r'Home|Residen|House|Apartment|Condo')
hc_all = get_remainder_of_two(pub_all, res_all)
hc_aed = get_remainder_of_two(pub_aed, res_aed)
hc_no = get_remainder_of_two(pub_no, res_no)
loc_p = calc_chi2_pval('Location Type', [r'Public|Street|Airport', r'Home|House', r'Healthcare|Other'])

wit_by_all, wit_by_aed, wit_by_no = get_cat('Arrest witnessed by', r'\bBystander\b|\bLay person\b|\bFamily\b')
wit_un_all = get_remainder_pct(wit_by_all)
wit_un_aed = get_remainder_pct(wit_by_aed)
wit_un_no = get_remainder_pct(wit_by_no)
wit_miss_all, wit_miss_aed, wit_miss_no = get_missing('Arrest witnessed by')
wit_p = calc_chi2_pval('Arrest witnessed by', [r'Bystander', r'Unwitnessed|No|None'])

# --- 3. BYSTANDER RESPONSE ---
cpr_all, cpr_aed, cpr_no = get_cat('Bystander CPR', r'\bYes\b')
cpr_miss_all, cpr_miss_aed, cpr_miss_no = get_missing('Bystander CPR')
cpr_p = calc_chi2_pval('Bystander CPR', [r'Yes', r'No'])

# --- 4. OPERATIONAL TIME ---
ems_time_all, ems_time_aed, ems_time_no, ems_time_p = get_cont('EMS_Response_Time')
shock_time_all, shock_time_aed, shock_time_no, shock_time_p = get_cont('Time_to_First_Shock')
dacpr_all, dacpr_aed, dacpr_no = get_cat('DA-CPR', r'\bYes\b')
time_miss_all, time_miss_aed, time_miss_no = get_missing('Time_to_First_Shock')
dacpr_p = calc_chi2_pval('DA-CPR', [r'Yes', r'No'])

# --- 5. CLINICAL OUTCOMES ---
died_ed_all, died_ed_aed, died_ed_no = get_cat('', r'Died in ED|Died at Scene', search_multiple_cols=True)
died_ward_all, died_ward_aed, died_ward_no = get_cat('', r'Died in the hospital|Died in ward', search_multiple_cols=True)
surv_dc_all, surv_dc_aed, surv_dc_no = get_cat('', r'Discharged Alive', search_multiple_cols=True)
surv_30_all, surv_30_aed, surv_30_no = get_cat('', r'Discharged Alive|Remains in hospital at 30th day', search_multiple_cols=True)
out_miss_all, out_miss_aed, out_miss_no = get_multi_missing(['Outcome of patient', 'Patient status', 'Final status at scene'])
remains_30_all, remains_30_aed, remains_30_no = get_cat('', r'Remains in hospital at 30th day', search_multiple_cols=True)

# --- 6. NEUROLOGICAL OUTCOMES ---
cpc_12_all, cpc_12_aed, cpc_12_no = get_cat('Patient neurological status - Cerebral', r'1|2|Good|Moderate')
cpc_34_all, cpc_34_aed, cpc_34_no = get_cat('Patient neurological status - Cerebral', r'3|4|Severe|Coma')
cpc_5_all, cpc_5_aed, cpc_5_no = get_cat('Patient neurological status - Cerebral', r'5|Brain death|Dead')
cpc_miss_all, cpc_miss_aed, cpc_miss_no = get_missing('Patient neurological status - Cerebral')

# Build and plot out table

In [18]:
table_data = [
    ["Variable", f"Overall Cohort (N = {n_total})", f"AED Applied (n = {n_aed})", f"AED Not Applied (n = {n_no_aed})", "p-value"],
    
    ["Demographics", "", "", "", ""],
    ["Age (Mean ± SD)", age_all, age_aed, age_no, age_p],
    ["Gender (Male %)", gen_all, gen_aed, gen_no, gen_p],
    
    ["Ethnicity (%)", "", "", "", eth_p],
    ["- Chinese", chi_all, chi_aed, chi_no, ""],
    ["- Malay", mal_all, mal_aed, mal_no, ""],
    ["- Indian", ind_all, ind_aed, ind_no, ""],
    ["- Other / Unknown¹", oth_all, oth_aed, oth_no, ""],
    
    ["Arrest Context", "", "", "", ""],
    ["Arrest Location (%)", "", "", "", loc_p],
    ["- Public Area", pub_all, pub_aed, pub_no, ""],
    ["- Residential Area", res_all, res_aed, res_no, ""],
    ["- Healthcare / Other", hc_all, hc_aed, hc_no, ""],
    
    ["Witness Status (%)", "", "", "", wit_p],
    ["- Bystander Witnessed", wit_by_all, wit_by_aed, wit_by_no, ""],
    ["- Unwitnessed", wit_un_all, wit_un_aed, wit_un_no, ""],
    
    ["Bystander Response", "", "", "", ""],
    ["Bystander CPR Performed (%)", cpr_all, cpr_aed, cpr_no, cpr_p],
    ["Dispatch-Assisted CPR (%)", dacpr_all, dacpr_aed, dacpr_no, dacpr_p],
    
    ["Operational Time (Minutes)", "", "", "", ""],
    ["EMS Response Time (Mean ± SD)", ems_time_all, ems_time_aed, ems_time_no, ems_time_p],
    ["Time to First Shock (Mean ± SD)²", shock_time_all, shock_time_aed, shock_time_no, shock_time_p],
    
    ["Clinical Outcomes (Sums to 100%)³", "", "", "", ""],
    ["- Died in Prehospital/ED Setting", died_ed_all, died_ed_aed, died_ed_no, ""],
    ["- Hospitalised & Died in Ward", died_ward_all, died_ward_aed, died_ward_no, ""],
    ["- Survived to Discharge", surv_dc_all, surv_dc_aed, surv_dc_no, ""],
    ["- Remained in Hospital at Day 30", remains_30_all, remains_30_aed, remains_30_no, ""],
    
    ["Neurological Outcomes (%)", "", "", "", ""],
    ["Good Neurological (CPC 1-2)", cpc_12_all, cpc_12_aed, cpc_12_no, ""],
    ["Bad Neurological (CPC 3-4)", cpc_34_all, cpc_34_aed, cpc_34_no, ""],
    ["Worst Neurological / Dead (CPC 5)", cpc_5_all, cpc_5_aed, cpc_5_no, ""]
]

# Create DataFrame
df_table1 = pd.DataFrame(table_data[1:], columns=table_data[0])

# Styling for Jupyter Display
styled_table = df_table1.style.hide(axis='index').set_properties(**{'text-align': 'left'})
display(styled_table)

# Print the Footnotes manually below the table
print("\nFOOTNOTES:")
print(f"¹ Other/Unknown includes Eurasian, Caucasian, and unspecified entries.")
print(f"² Time data was 99.3% complete; {time_miss_all} of cases with missing timestamps were excluded from means.")
print(f"³ Clinical outcomes and CPC scores were 100% complete for the final analytic cohort.")

Variable,Overall Cohort (N = 2076),AED Applied (n = 129),AED Not Applied (n = 1947),p-value
Demographics,,,,
Age (Mean ± SD),61.5 ± 14.1,59.6 ± 13.5,61.6 ± 14.1,0.099
Gender (Male %),83.1%,90.7%,82.6%,0.024
Ethnicity (%),,,,0.577
- Chinese,66.5%,62.8%,66.7%,
- Malay,17.0%,18.6%,16.8%,
- Indian,11.9%,15.5%,11.7%,
- Other / Unknown¹,4.4%,3.1%,4.5%,
Arrest Context,,,,
Arrest Location (%),,,,<0.001



FOOTNOTES:
¹ Other/Unknown includes Eurasian, Caucasian, and unspecified entries.
² Time data was 99.3% complete; 0.7% of cases with missing timestamps were excluded from means.
³ Clinical outcomes and CPC scores were 100% complete for the final analytic cohort.


In [19]:
# 1. Define the output path
tables_dir = "../Survival_Curve_Analysis/results/tables"
os.makedirs(tables_dir, exist_ok=True)

# 2. Save as CSV (Best for raw data/version control)
csv_path = os.path.join(tables_dir, "Table_1_Baseline_Characteristics.csv")
df_table1.to_csv(csv_path, index=False)

# 3. Save as Excel (Best for sharing with the HSRC team or Dr. Sean)
excel_path = os.path.join(tables_dir, "Table_1_Baseline_Characteristics.xlsx")
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_table1.to_excel(writer, index=False, sheet_name='Table 1')
    
print(f"Success! Table 1 saved to: {tables_dir}")

Success! Table 1 saved to: ../Survival_Curve_Analysis/results/tables


# Appendix Table for Missingness

In [20]:
# Appendix Table Generation
def generate_appendix():
    appendix_data = [
        ["Age", "Age", "100.0%", 0],
        ["Gender", "Gender", "100.0%", 0],
        ["Ethnicity", "Race", "100.0%", 0],
        ["Arrest Location", "Location Type", "100.0%", 0],
        ["Witness Status", "Arrest witnessed by", "100.0%", 0],
        ["Bystander CPR", "Bystander CPR", "100.0%", 0],
        ["EMS Response Time", "Call_Time, EMS_Time", "100.0%", 0],
        ["Time to First Shock", "Call_Time, Shock_Time", time_miss_all.replace("%", " (Remaining)"), "15"],
        ["Survival Outcomes", "Outcome of patient, Patient status", "100.0%", 0],
        ["Neurological Status", "Cerebral Performance Category (CPC)", "100.0%", 0]
    ]
    
    df_appendix = pd.DataFrame(appendix_data, columns=["Variable", "Source Column(s)", "Completeness (%)", "Missing/Unknown (n)"])
    
    # Save to your results folder
    appendix_path = os.path.join("../Survival_Curve_Analysis/results/tables", "Appendix_Table_A1_Missing_Data.csv")
    df_appendix.to_csv(appendix_path, index=False)
    
    return df_appendix

display(generate_appendix())

,Variable,Source Column(s),Completeness (%),Missing/Unknown (n)
0,Age,Age,100.0%,0
1,Gender,Gender,100.0%,0
2,Ethnicity,Race,100.0%,0
3,Arrest Location,Location Type,100.0%,0
4,Witness Status,Arrest witnessed by,100.0%,0
5,Bystander CPR,Bystander CPR,100.0%,0
6,EMS Response Time,"Call_Time, EMS_Time",100.0%,0
7,Time to First Shock,"Call_Time, Shock_Time",0.7 (Remaining),15
8,Survival Outcomes,"Outcome of patient, Patient status",100.0%,0
9,Neurological Status,Cerebral Performance Category (CPC),100.0%,0
